# Clasificador por casilla v2 — entrenamiento en Colab

Este notebook entrena **(b) el clasificador de 13 clases** de la arquitectura v2:
las 12 piezas + `vacia`.

| | |
|---|---|
| Modelo | `mobilenet_v3_small` con `weights=DEFAULT` (ImageNet) |
| Entrada | casilla rectificada 100x100 -> redimensionada a 128x128 |
| Clases | 13: los 12 nombres de `classes.txt` + `vacia` (id 12) |
| Paquete | `clasificador_v2.zip` (406.7 MB) en `Mi unidad/ajedrez_v2/` |
| Train | 115,584 casillas de 1,806 tableros compuestos |
| Valid | 19,072 casillas (238 tableros: 226 compuestos + 12 recortes reales) |
| Test | 17,536 casillas (234 tableros: 226 compuestos + 8 recortes reales) |

**Por que este modelo:** YOLO detecta cajas e ignora que el tablero es una rejilla
conocida de 64 casillas. Aqui se rectifica el tablero, se recortan las 64 casillas
y se clasifica cada una. El split se hace **por tablero, nunca por casilla**: las
64 casillas de una misma foto caen siempre en el mismo split, si no hay fuga.

Antes de ejecutar: **Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU**.

## 1. Verificar la GPU

In [ ]:
#@title 1. Verificar GPU y versiones
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: ve a Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")

## 2. Dependencias

`torch`, `torchvision` y `Pillow` ya vienen en Colab. La celda solo instala si
falta algo, y **nunca** reinstala `torch`: cambiarlo rompe la GPU de la sesion.

In [ ]:
#@title 2. Comprobar dependencias sin tocar torch
import importlib.util
import subprocess
import sys

for paquete, modulo in (("torchvision", "torchvision"), ("pillow", "PIL")):
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import torch
import torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

## 3. Traer el paquete de datos

El zip trae:

```
casillas/<clase>/<archivo>.jpg     152,192 recortes de 100x100
split/train.txt, valid.txt, test.txt   una ruta relativa por linea
labels/clases.txt                    las 13 clases en orden
labels/*.csv                         los CSV de referencia del etiquetado
```

La lista de cada split es la que manda: no hay `ImageFolder` ni carpetas por
split. Se descomprime en `/content` (disco local), no en Drive: 152 mil archivos
pequeños leidos desde Drive son lentisimos.

**Sube `clasificador_v2.zip` a `Mi unidad/ajedrez_v2/`** y deja `USAR_DRIVE = True`.

In [ ]:
#@title 3. Descomprimir clasificador_v2.zip y contar por split
import shutil
import zipfile
from pathlib import Path

USAR_DRIVE = True
ZIP_EN_DRIVE = Path("/content/drive/MyDrive/ajedrez_v2/clasificador_v2.zip")
DESTINO = Path("/content/ajedrez_v2/clasificador")
ESPERADO = {"train": 115584, "valid": 19072, "test": 17536}

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    origen = ZIP_EN_DRIVE
else:
    from google.colab import files
    subidos = files.upload()
    origen = Path(next(iter(subidos)))

if not origen.exists():
    raise FileNotFoundError(
        f"No encuentro {origen}. Sube clasificador_v2.zip a Mi unidad/ajedrez_v2/ "
        "o pon USAR_DRIVE = False."
    )

if DESTINO.exists():
    shutil.rmtree(DESTINO)
DESTINO.mkdir(parents=True)

with zipfile.ZipFile(origen) as zf:
    zf.extractall(DESTINO)

print("Contenido:", sorted(p.name for p in DESTINO.iterdir()))
for split, esperado in ESPERADO.items():
    n = sum(1 for linea in (DESTINO / "split" / f"{split}.txt").read_text().splitlines()
            if linea.strip())
    print(f"  {split:5s} rutas={n:6d} (esperado {esperado:6d}) "
          f"{'OK' if n == esperado else 'REVISAR'}")

CLASES = [l.strip() for l in (DESTINO / "labels" / "clases.txt").read_text().splitlines()
          if l.strip()]
print("Clases (13):", CLASES)
assert len(CLASES) == 13 and CLASES[12] == "vacia", "labels/clases.txt no tiene 13 clases"

PROYECTO = Path("/content/drive/MyDrive/ajedrez_v2/clasificador")
PROYECTO.mkdir(parents=True, exist_ok=True)
print("Salida:", PROYECTO)

## 4. Dataset, transformaciones y pesos de clase

Un `Dataset` minimo que lee la lista de rutas del `.txt` y saca la clase del
**nombre de la carpeta** (`casillas/<clase>/archivo.jpg`). Nada de indices
guardados: la lista de split es la unica fuente de verdad.

- `Resize(128, 128)` + `Normalize` con las medias de ImageNet (el modelo es
  preentrenado, espera esa normalizacion).
- Solo en train, un `ColorJitter` suave: las dos sesiones de luz se parecen mas.
- **Pesos de clase** = frecuencia inversa sobre el split de train. `vacia` es el
  63.8% de las casillas; sin el peso el modelo aprende a decir "vacia" y listo.

In [ ]:
#@title 4. Datasets, loaders y pesos de clase
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

RAIZ = DESTINO
TAMANO = 128
BATCH = 128
EPOCAS = 12
LR = 1e-3
SEMILLA = 42
NUM_WORKERS = 4

torch.manual_seed(SEMILLA)

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((TAMANO, TAMANO)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((TAMANO, TAMANO)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class Casillas(Dataset):
    """Casillas de un split, leidas desde su .txt (rutas relativas al zip)."""

    def __init__(self, raiz: Path, txt: Path, clases: list[str], transform):
        self.raiz = raiz
        self.clases = clases
        self.transform = transform
        self.rutas = [l.strip() for l in txt.read_text(encoding="utf-8").splitlines()
                      if l.strip()]

    def __len__(self) -> int:
        return len(self.rutas)

    def __getitem__(self, indice: int):
        relativa = self.rutas[indice]
        # casillas/<clase>/<archivo>.jpg
        clase = relativa.split("/")[1]
        imagen = Image.open(self.raiz / relativa).convert("RGB")
        return self.transform(imagen), self.clases.index(clase)


conjuntos = {
    split: Casillas(RAIZ, RAIZ / "split" / f"{split}.txt", CLASES,
                    TRANSFORM_TRAIN if split == "train" else TRANSFORM_EVAL)
    for split in ("train", "valid", "test")
}
cargadores = {
    "train": DataLoader(conjuntos["train"], batch_size=BATCH, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True),
    "valid": DataLoader(conjuntos["valid"], batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True),
    "test": DataLoader(conjuntos["test"], batch_size=BATCH, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True),
}

conteo_train = Counter(ruta.split("/")[1] for ruta in conjuntos["train"].rutas)
faltan = [c for c in CLASES if conteo_train[c] == 0]
if faltan:
    raise SystemExit(f"Clases sin ejemplos en train: {faltan}. Revisa split/train.txt.")

totales = sum(conteo_train.values())
pesos = torch.tensor(
    [totales / (len(CLASES) * conteo_train[c]) for c in CLASES], dtype=torch.float32
)
pesos = pesos / pesos.mean()  # media 1: no mueve la escala del learning rate

print(f"{'clase':12s} {'train':>8s} {'%':>7s} {'peso':>6s}")
for clase, peso in zip(CLASES, pesos.tolist()):
    n = conteo_train[clase]
    print(f"{clase:12s} {n:8d} {100 * n / totales:6.2f}% {peso:6.2f}")
print(f"\nvacia = {100 * conteo_train['vacia'] / totales:.2f}% del train | "
      f"dama_b/vacia = {conteo_train['dama_b'] / conteo_train['vacia']:.3f}")
print("items:", {k: len(v) for k, v in conjuntos.items()})

## 5. Modelo, optimizador y funcion de perdida

`mobilenet_v3_small` preentrenado en ImageNet: 2.5 M de parametros con la cabeza
de 1000 clases, ~1.5 M al cambiarla por la de 13. Entra de sobra en una T4 y en el
ESP32/edge del pipeline. La cabeza es una `Linear` de 13 salidas que reemplaza la
de 1000 clases.

`AdamW` con `lr = 1e-3` + cosine, y `CrossEntropyLoss(weight=...)` con los pesos
calculados arriba.

In [ ]:
#@title 5. Crear el modelo
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"


def crear_modelo(n_clases: int) -> nn.Module:
    try:
        red = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    except (AttributeError, TypeError):  # torchvision viejo
        red = models.mobilenet_v3_small(pretrained=True)
    for parametro in red.features.parameters():
        parametro.requires_grad = True
    entrada = red.classifier[3].in_features
    red.classifier[3] = nn.Linear(entrada, n_clases)
    return red


modelo = crear_modelo(len(CLASES)).to(DISPOSITIVO)
optimizador = AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
planificador = CosineAnnealingLR(optimizador, T_max=EPOCAS)
criterio = nn.CrossEntropyLoss(weight=pesos.to(DISPOSITIVO))

n_parametros = sum(p.numel() for p in modelo.parameters())
print("Dispositivo:", DISPOSITIVO)
print(f"Parametros: {n_parametros / 1e6:.2f} M")
print("Salida:", modelo.classifier[3].out_features, "clases")

## 6. Entrenar (12 epocas)

Se guarda en Drive cada vez que `valid` mejora. Si Colab se desconecta, el
`best_clasificador.pt` sigue en tu unidad.

In [ ]:
#@title 6. Entrenar
import copy
import time
from pathlib import Path

import torch


def evaluar(red, cargador) -> tuple[float, float]:
    red.eval()
    perdida_total = 0.0
    aciertos = 0
    vistos = 0
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            imagenes = imagenes.to(DISPOSITIVO, non_blocking=True)
            etiquetas = etiquetas.to(DISPOSITIVO, non_blocking=True)
            salida = red(imagenes)
            perdida_total += criterio(salida, etiquetas).item() * len(etiquetas)
            aciertos += (salida.argmax(1) == etiquetas).sum().item()
            vistos += len(etiquetas)
    return perdida_total / max(vistos, 1), aciertos / max(vistos, 1)


SALIDA = PROYECTO
SALIDA.mkdir(parents=True, exist_ok=True)
mejor_acc = 0.0
historial = []

for epoca in range(1, EPOCAS + 1):
    arranque = time.time()
    modelo.train()
    perdida_total = 0.0
    aciertos = 0
    vistos = 0
    for imagenes, etiquetas in cargadores["train"]:
        imagenes = imagenes.to(DISPOSITIVO, non_blocking=True)
        etiquetas = etiquetas.to(DISPOSITIVO, non_blocking=True)
        optimizador.zero_grad(set_to_none=True)
        salida = modelo(imagenes)
        perdida = criterio(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item() * len(etiquetas)
        aciertos += (salida.argmax(1) == etiquetas).sum().item()
        vistos += len(etiquetas)
    planificador.step()

    perdida_train = perdida_total / max(vistos, 1)
    acc_train = aciertos / max(vistos, 1)
    perdida_val, acc_val = evaluar(modelo, cargadores["valid"])
    segundos = time.time() - arranque
    historial.append((epoca, perdida_train, acc_train, perdida_val, acc_val))
    print(f"epoca {epoca:2d}/{EPOCAS}  loss={perdida_train:.4f} train={acc_train:.3f} "
          f"valid={acc_val:.3f}  ({segundos:.0f}s)")

    if acc_val > mejor_acc:
        mejor_acc = acc_val
        torch.save({
            "modelo": copy.deepcopy(modelo.state_dict()),
            "clases": CLASES,
            "epoca": epoca,
            "acc_valid": acc_val,
            "tamano": TAMANO,
        }, SALIDA / "best_clasificador.pt")
        print(f"  -> mejor hasta ahora (valid={acc_val:.3f}), guardado en Drive")

print()
print(f"Mejor accuracy en valid: {mejor_acc:.4f}")
print("Pesos:", SALIDA / "best_clasificador.pt")

## 7. Matriz de confusion por clase en `valid` y `test`

El accuracy global esconde el problema real: `vacia` es el 62% de las casillas. Si
el modelo solo acierta `vacia` y las piezas comunes, el numero se ve bien y el
pipeline falla. Aqui se ve clase por clase.

In [ ]:
#@title 7. Matriz de confusion y reporte por clase
import numpy as np
import torch


def matriz_confusion(red, cargador, n_clases: int) -> np.ndarray:
    matriz = np.zeros((n_clases, n_clases), dtype=np.int64)
    red.eval()
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            salida = red(imagenes.to(DISPOSITIVO)).argmax(1).cpu().numpy()
            for real, pred in zip(etiquetas.numpy(), salida):
                matriz[real, pred] += 1
    return matriz


if not (SALIDA / "best_clasificador.pt").exists():
    raise SystemExit("No hay best_clasificador.pt: ejecuta la celda 6 primero")

pesos_guardados = torch.load(SALIDA / "best_clasificador.pt", map_location=DISPOSITIVO)
modelo.load_state_dict(pesos_guardados["modelo"])

for split in ("valid", "test"):
    matriz = matriz_confusion(modelo, cargadores[split], len(CLASES))
    total = matriz.sum()
    aciertos = np.trace(matriz)
    print(f"== {split} ==")
    print(f"   accuracy global: {aciertos / total:.4f}  ({aciertos}/{total})")
    print(f"   {'clase':12s} {'n':>7s} {'prec':>6s} {'recall':>6s}")
    for i, clase in enumerate(CLASES):
        n = matriz[i].sum()
        if n == 0:
            print(f"   {clase:12s} {0:7d}   (sin ejemplos)")
            continue
        recall = matriz[i, i] / n
        predichos = matriz[:, i].sum()
        precision = matriz[i, i] / predichos if predichos else 0.0
        print(f"   {clase:12s} {n:7d} {precision:6.3f} {recall:6.3f}")

    errores = []
    for real in range(len(CLASES)):
        for pred in range(len(CLASES)):
            if real != pred and matriz[real, pred] > 0:
                errores.append((int(matriz[real, pred]), CLASES[real], CLASES[pred]))
    errores.sort(reverse=True)
    print("   confusiones mas frecuentes:")
    for cantidad, real, pred in errores[:8]:
        print(f"     {real:12s} -> {pred:12s} {cantidad}")
    print()

## 8. Guardar el modelo final en Drive

`best_clasificador.pt` guarda el `state_dict`, la lista de clases y el tamano de
entrada. Para usarlo despues hay que reconstruir `mobilenet_v3_small` y cargar el
`state_dict`: el archivo no trae la arquitectura, solo los pesos.

In [ ]:
#@title 8. Verificar que el modelo se puede recargar y listar Drive
import shutil

import torch

SALIDA = PROYECTO / "entrega"
SALIDA.mkdir(parents=True, exist_ok=True)

shutil.copy2(PROYECTO / "best_clasificador.pt", SALIDA / "best_clasificador.pt")
(SALIDA / "clases.txt").write_text("\n".join(CLASES) + "\n", encoding="utf-8")

comprobacion = crear_modelo(len(CLASES)).to(DISPOSITIVO)
comprobacion.load_state_dict(torch.load(SALIDA / "best_clasificador.pt",
                                        map_location=DISPOSITIVO)["modelo"])
comprobacion.eval()
print("Recarga OK:", SALIDA / "best_clasificador.pt")
for p in sorted(SALIDA.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / 1024:8.1f} KB")

## Como entra esto en el pipeline v2

Una vez entrenado, el pipeline (`src/`) hace, en este orden:

1. **Detector YOLO** (`best.pt` del otro notebook): encuentra las cajas de las
   fichas en el frame completo.
2. **Rectificacion del tablero**: con las 4 esquinas se aplica la homografia y el
   tablero queda en 800x800 con casillas de 100x100.
3. **Clasificador por casilla** (`best_clasificador.pt`): cada una de las 64
   casillas se recorta, se redimensiona a 128x128 y se clasifica en una de las 13
   clases. `vacia` no es un fracaso: es informacion.
4. **Filtro de movimientos legales con `python-chess`**: se construye el FEN con
   las 64 predicciones y se descartan las posiciones que un tablero legal no
   permite (dos reyes por bando, peones en la primera fila, etc.). Ese post-filtro
   es el que convierte "64 predicciones sueltas" en una posicion de ajedrez.

La comparacion de T7 (**YOLO solo vs clasificador solo vs combinado**) se hace
sobre el **mismo test real**: los recortes reales de `test` son exactamente los
mismos que usa el detector, porque el reparto se hizo por imagen fuente.

## Si algo falla

| Sintoma | Causa | Solucion |
|---|---|---|
| `SIN GPU` | Entorno sin GPU | Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU |
| `No encuentro ...clasificador_v2.zip` | Ruta de Drive distinta | Revisa `ZIP_EN_DRIVE` o pon `USAR_DRIVE = False` |
| Descomprimir tarda mucho | 152 mil archivos pequeños | Es normal la primera vez: 3-5 minutos |
| `CUDA out of memory` | `BATCH` alto | Baja `BATCH` a 64 |
| `accuracy` alta pero el pipeline falla | El modelo dice `vacia` a todo | Mira la matriz por clase: si `recall` de piezas es bajo, sube epocas o el peso de clase |
| DataLoader lento | `/content` lleno o `NUM_WORKERS` bajo | Sube `NUM_WORKERS` a 8 y vuelve a descomprimir |
| Se perdio el entrenamiento | Colab cerro la sesion | El `best_clasificador.pt` esta en Drive: no hay nada que rehacer |